# VMR Project — Data Exploration
Use this notebook to explore QVHighlights dataset, inspect features, and prototype analysis.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

DATA_DIR = Path('../data/qvhighlights')
ANNO_DIR = DATA_DIR / 'annotations'

## 1. Load Annotations

In [ ]:
def load_jsonl(path):
    data = []
    with open(path) as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

train_data = load_jsonl(ANNO_DIR / 'highlight_train_release.jsonl')
val_data = load_jsonl(ANNO_DIR / 'highlight_val_release.jsonl')

print(f'Train: {len(train_data)} samples')
print(f'Val: {len(val_data)} samples')
print()
print('Sample entry keys:', list(train_data[0].keys()))

In [ ]:
# Inspect a sample
sample = train_data[0]
for k, v in sample.items():
    if isinstance(v, list) and len(v) > 5:
        print(f'{k}: [{v[0]}, {v[1]}, ...] (len={len(v)})')
    else:
        print(f'{k}: {v}')

## 2. Dataset Statistics

In [ ]:
# Video durations
train_durations = [item['duration'] for item in train_data]
val_durations = [item['duration'] for item in val_data]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train_durations, bins=30, color='#2E5A88', edgecolor='white', alpha=0.8)
axes[0].set_title('Training Video Durations')
axes[0].set_xlabel('Duration (seconds)')
axes[1].hist(val_durations, bins=30, color='#E8634A', edgecolor='white', alpha=0.8)
axes[1].set_title('Validation Video Durations')
axes[1].set_xlabel('Duration (seconds)')
plt.tight_layout()
plt.show()

print(f'Train duration — mean: {np.mean(train_durations):.1f}s, std: {np.std(train_durations):.1f}s')
print(f'Val duration   — mean: {np.mean(val_durations):.1f}s, std: {np.std(val_durations):.1f}s')

In [ ]:
# Query length distribution
train_qlens = [len(item['query'].split()) for item in train_data]
plt.hist(train_qlens, bins=20, color='#2E5A88', edgecolor='white', alpha=0.8)
plt.title('Query Length Distribution (Train)')
plt.xlabel('Number of words')
plt.ylabel('Count')
plt.show()
print(f'Mean query length: {np.mean(train_qlens):.1f} words')

In [ ]:
# Moment position analysis (for bias analysis)
centers = []
durations_norm = []
for item in train_data:
    vid_dur = item['duration']
    for w in item.get('relevant_windows', []):
        s, e = w
        centers.append(((s + e) / 2) / vid_dur)
        durations_norm.append((e - s) / vid_dur)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(centers, bins=30, color='#2E5A88', edgecolor='white', alpha=0.8)
axes[0].set_title('Moment Center Position (normalized)')
axes[0].set_xlabel('Position in video (0=start, 1=end)')
axes[0].axvline(0.5, color='red', linestyle='--', alpha=0.5, label='Center')
axes[0].legend()

axes[1].hist(durations_norm, bins=30, color='#E8634A', edgecolor='white', alpha=0.8)
axes[1].set_title('Moment Duration (normalized)')
axes[1].set_xlabel('Fraction of video')
plt.tight_layout()
plt.show()

print(f'Mean center: {np.mean(centers):.3f} (0.5 = perfectly centered)')
print(f'Mean duration ratio: {np.mean(durations_norm):.3f}')

## 3. Feature Inspection

In [ ]:
# Check what features are available
import glob
feat_dir = DATA_DIR / 'features'
print('Feature files:')
for f in sorted(glob.glob(str(feat_dir / '**/*'), recursive=True))[:20]:
    print(f'  {f}')

In [ ]:
# Load and inspect a feature file
# Uncomment and adjust path based on your feature format

# import h5py
# with h5py.File(feat_dir / 'clip_features.h5', 'r') as f:
#     print('Keys:', list(f.keys())[:5])
#     sample_key = list(f.keys())[0]
#     print(f'Sample shape ({sample_key}): {f[sample_key].shape}')
#     print(f'Sample dtype: {f[sample_key].dtype}')

## 4. Quick spaCy Test (for linguistic analysis)

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')

# Test on a few queries
for item in train_data[:5]:
    query = item['query']
    doc = nlp(query)
    verbs = [t.text for t in doc if t.pos_ == 'VERB']
    nouns = [t.text for t in doc if t.pos_ in ('NOUN', 'PROPN')]
    print(f'Query: {query}')
    print(f'  Verbs: {verbs}   Nouns: {nouns}')
    print()